# 08 - Debugging and reasoning eval


## Goal

Run the hand-authored eval set against the graph-only RAG baseline and render the Markdown report inline. This is the harness every fine-tune in this lab must beat.


## Prerequisites

- Notebooks 02-04 read; you should know what `find_by_name`, `trace`, and `answer` do.
- No model weights involved — the baseline answerer is deterministic graph code.


## Environment bootstrap

This cell makes the notebook portable between a local checkout and Colab.

- **Local**: when the notebook lives inside the repo, we add the repo root to `sys.path`
  so the `src` package imports cleanly.
- **Colab**: the import will fail with `ModuleNotFoundError`. We catch that and print a
  one-line reminder showing the `git clone` the learner should run. We deliberately do
  **not** execute the clone for them — the lab policy is *recipes only, no auto-downloads*.


In [ ]:
import sys
from pathlib import Path

try:
    # Local checkout: walk up from the notebook to the repo root.
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "common" / "paths.py").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            break
    from src.common.paths import REPO_ROOT, MINI_REPO, ensure_dirs
    ensure_dirs()
    print(f"repo root: {REPO_ROOT}")
    print(f"mini repo: {MINI_REPO}")
except ModuleNotFoundError:
    print("`src` not importable. If you are on Colab, run this in a separate cell:")
    print("    !git clone https://example.invalid/kde_ontology_slm_lab.git")
    print("    %cd kde_ontology_slm_lab")
    print("Then re-run this cell. We will not auto-clone for you (lab policy:")
    print("recipes only, no auto-downloads).")


## 1. Build the graph


In [ ]:
from src.common.paths import MINI_REPO, EVAL_DIR
from src.repo_ingest.scanner import scan
from src.repo_ingest.cmake_reader import read_cmake
from src.repo_ingest.cpp_reader import read_cpp
from src.repo_ingest.qml_reader import read_qml
from src.repo_ingest.dbus_reader import read_dbus
from src.repo_ingest.kconfig_reader import read_kconfig
from src.repo_ingest.desktop_file_reader import read_desktop
from src.repo_ingest.log_reader import read_log
from src.ontology.extractor import (
    ExtractionBundle, from_cmake, from_cpp, from_qml, from_dbus,
    from_kconfig, from_desktop, from_log,
)
from src.ontology.schema import Entity
from src.common.ids import make_id
from src.graph.builder import build_graph

rep = scan(MINI_REPO)
b = ExtractionBundle()
rid = b.add_entity(Entity(id=make_id('Repository', MINI_REPO.name),
                          type='Repository', name=MINI_REPO.name,
                          source_path=str(MINI_REPO)))
for sf in rep.by_kind('cmake'): from_cmake(b, read_cmake(sf.path), rid)
for sf in rep.by_kind('cpp_header') + rep.by_kind('cpp_source'): from_cpp(b, read_cpp(sf.path))
for sf in rep.by_kind('qml'): from_qml(b, read_qml(sf.path))
for sf in rep.by_kind('dbus'): from_dbus(b, read_dbus(sf.path))
for sf in rep.by_kind('kconfig'): from_kconfig(b, read_kconfig(sf.path))
for sf in rep.by_kind('desktop'): from_desktop(b, read_desktop(sf.path))
for sf in rep.by_kind('log'): from_log(b, read_log(sf.path))
g = build_graph(b)


## 2. Load the eval set

`mini_repo_eval_set()` returns six items spanning architecture, code navigation, debugging, tool use, and refusal. Each item declares `must_mention` (and sometimes `must_not_mention`) strings the grader checks against the answer.


In [ ]:
from src.eval.eval_set_builder import mini_repo_eval_set

items = mini_repo_eval_set()
for it in items:
    print(f"{it['id']:30s} [{it['category']:18s}] {it['question']}")


## 3. Answer each question with the RAG baseline


In [ ]:
from src.rag.answer_with_evidence import answer

answers = []
for it in items:
    a = answer(g, it['question'], k=6)
    answers.append((it, a))
    print(f"\n--- {it['id']} ---")
    print(a.text)


## 4. Grade and aggregate

The grader (`src/eval/answer_grader.py`) checks substring presence. Aggregation (`src/eval/report.py`) groups by category and computes pass rate + mean recall.


In [ ]:
from src.eval.answer_grader import grade
from src.eval.report import aggregate, to_markdown, save

grades = [grade(a.text, it) for it, a in answers]
for it, gr in zip(items, grades):
    flag = 'PASS' if gr.passed else 'FAIL'
    print(f"  {flag}  {it['id']:30s} recall={gr.mention_recall:.0%}"
          f" forbidden_hit={gr.forbidden_hit}")

rep_data = aggregate(grades)
print('\noverall pass rate:', f"{rep_data['overall_pass_rate']:.1%}")


## 5. Render the Markdown report inline

`to_markdown(report)` returns a string we can display with `IPython.display.Markdown` for a nice in-notebook table. We also persist the JSON + Markdown copies under `artifacts/eval_reports/`.


In [ ]:
md = to_markdown(rep_data)
try:
    from IPython.display import Markdown, display
    display(Markdown(md))
except ImportError:
    print(md)

jp, mp = save(rep_data, EVAL_DIR, name='mini_repo_eval')
print('json :', jp)
print('md   :', mp)


## Summary

You ran the canonical eval suite end-to-end against the RAG baseline. Any fine-tuned model you train on the dataset from notebook 06 should be measured against the same harness. Notebook 09 compares the two paradigms side-by-side.


## Exercises

1. Add a seventh eval item that asks about a class that does **not** exist. Make sure the refusal path still triggers.
2. Categorise the per-question failures. Which category has the lowest recall, and what additional extraction would lift it?
3. Plot the per-category pass-rate as a bar chart (matplotlib).
